# Real Estate — Feature Selection

Preprocessing produced a wide matrix, most of it dummy columns. This notebook prunes it in two
passes — **importance** first, keeping what a gradient-boosted model actually splits on, then
**VIF**, iteratively removing what is redundant given the rest — and then grid-searches the model
families on what survives.

The question it answers is whether a much smaller feature set costs anything in dollars of error.

## Learning objectives

- Rank features by tree importance and keep the useful ones
- Iteratively prune collinear features by variance inflation factor
- Explain why importance and VIF remove different columns
- Grid-search model families on the reduced feature set and compare MAE against the full-feature run

## Background

This notebook assumes the preprocessed table from `U1_RealEstate-2_Preprocess`, the model families
and metrics of `U1_RealEstate-3_Regression` — including its argument for selecting on **MAE** rather
than $R^2$ — and the two pruning tools from `U1-4_FeatSelect-1_VIF` and
`U1-4_FeatSelect-2_FeatureImportance`.

**Prerequisites:** `U1_RealEstate-3_Regression` (models, metrics, why MAE),
`U1-4_FeatSelect-1_VIF`, `U1-4_FeatSelect-2_FeatureImportance`

**Dataset:** `Real Estate Data_preprocessed.csv`.

**References:** https://scikit-learn.org/stable/modules/feature_selection.html

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from tqdm import tqdm

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)
pd.set_option("display.precision", 4)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load preprocessed data

Notebook 2 (`U1_RealEstate-2_Preprocess`) already cleaned this dataset, engineered the features, and encoded every column — saving the finished result to `Real Estate Data_preprocessed.csv`. Rather than repeat all of that preprocessing here, we load that file and go straight to preparing the data for modeling.

In [2]:
data_folder = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'

# Load the fully preprocessed dataset saved by notebook 2.
df = pd.read_csv(data_folder + 'Real Estate/Real Estate Data_preprocessed.csv')

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1256 entries, 0 to 1255
Columns: 195 entries, Type to Sale Condition_Sale between family members
dtypes: float64(4), int64(167), object(24)
memory usage: 1.9+ MB


,Type,Zoning Class,Lot Frontage,Lot Area,Lot Shape,Land Contour,Lot Config,Land Slope,Nbhd,Location Condition,Bldg Type,House Style,Overall Qual,Overall Cond,Year,Year Remod Add,Roof Style,Roof Material,Exterior Primary,Masonry/Veneer Area,Exterior Qual,Exterior Cond,Foundation,Basement Height,Basement Cond,Basement Exposure,Basement Finish,Heating Qual,Central Air,Electrical,Basement Full Baths,Basement Half baths,Full Baths Above Grade,Half Baths Above Grade,Bedrooms Above Grade,Kitchens Above Grade,Kitchen Qual,Total Rooms Above Grade,Functionality,Fireplaces,Fireplce Qual,Garage Type,Year Garage,Garage Finish,Garage Cars,Garage Qual,Garage Cond,Paved Drive,Pool Qual,Fence,...,Exterior Primary_Stucco,Exterior Primary_Wood Shingles,Exterior Primary_Wood Siding,Foundation_Brick & Tile,Foundation_Cinder Block,Foundation_Slab,Foundation_Stone,Foundation_Wood,Basement Exposure_Avg Exposure,Basement Exposure_Good Exposure,Basement Exposure_Min Exposure,Basement Exposure_No Basement,Basement Finish_Avg Living Quarters,Basement Finish_Avg Rec Room,Basement Finish_Below Avg Living Quarters,Basement Finish_Good Living Quarters,Basement Finish_Low Quality,Basement Finish_No Basement,Electrical_60 AMP Fuse Box and mostly Romex wiring (Fair),Electrical_Fuse Box over 60 AMP and all Romex wiring (Average),Functionality_Major Deductions 1,Functionality_Minor Deductions 1,Functionality_Minor Deductions 2,Functionality_Moderate Deductions,Garage Type_Basement Garage,Garage Type_Built-In (Garage with Room),Garage Type_Car Port,Garage Type_Detached from home,Garage Type_More than one type of garage,Garage Type_No Garage,Garage Finish_Finished,Garage Finish_No Garage,Garage Finish_Rough Finished,Paved Drive_Dirt/Gravel,Paved Drive_Partial Pavement,Fence_Good Privacy,Fence_Good Wood,Fence_Minimum Privacy,Fence_Minimum Wood/Wire,Sale Type_Contract 15% Down payment regular terms,Sale Type_Contract Low Down,Sale Type_Contract Low Down payment and low interest,Sale Type_Contract Low Interest,Sale Type_Court Officer Deed/Estate,Sale Type_Home just constructed and sold,Sale Type_Warranty Deed - Cash,"Sale Condition_Abnormal Sale - trade, foreclosure, short sale","Sale Condition_Allocation - two linked properties with separate deeds, typically condo with a garage unit",Sale Condition_Home was not completed when last assessed (associated with New Homes),Sale Condition_Sale between family members
0,2-STORY 1946 & NEWER,Resid Low Density,65.0,8450,Regular,Level,Inside lot,Gentle,College Creek,Normal,1-family Detached,2 story,7,5,2003,2003,Gable,Composite Shingle,Vinyl Siding,196.0,3,2,Poured Contrete,3,2,No Exposure,Good Living Quarters,4,1,Standard Circuit Breakers & Romex,1,0,2,1,3,1,3,8,Typical Functionality,0,0,Attached to home,2003.0,Rough Finished,2,2,2,Paved,0,No Fence,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1-STORY 1946 & NEWER,Resid Low Density,80.0,9600,Regular,Level,Frontage on 2 sides,Gentle,Veenker,Adjacent Feeder St,1-family Detached,1 story,6,8,1976,1976,Gable,Composite Shingle,Metal Siding,0.0,2,2,Cinder Block,3,2,Good Exposure,Avg Living Quarters,4,1,Standard Circuit Breakers & Romex,0,1,2,0,3,1,2,6,Typical Functionality,1,2,Attached to home,1976.0,Rough Finished,2,2,2,Paved,0,No Fence,...,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2-STORY 1946 & NEWER,Resid Low Density,68.0,11250,Slightly irregular,Level,Inside lot,Gentle,College Creek,Normal,1-family Detached,2 story,7,5,2001,2002,Gable,Composite Shingle,Vinyl Siding,162.0,3,2,Poured Contrete,3,2,Min Exposure,Good Living Quarters,4,1,Standard Circuit Breakers & Romex,1,0,2,1,3,1,3,6,Typical Functionality,1,2,Attached to home,2001.0,Rough Finished,2,2,2,Paved,0,No Fence,...,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2-STORY 1945 & OLDER,Resid Low Density,60.0,9550,Slightly irregular,Level,Corner lot,Gentle,

## 2. Preprocess data

In [3]:
df.sample(5)

,Type,Zoning Class,Lot Frontage,Lot Area,Lot Shape,Land Contour,Lot Config,Land Slope,Nbhd,Location Condition,Bldg Type,House Style,Overall Qual,Overall Cond,Year,Year Remod Add,Roof Style,Roof Material,Exterior Primary,Masonry/Veneer Area,Exterior Qual,Exterior Cond,Foundation,Basement Height,Basement Cond,Basement Exposure,Basement Finish,Heating Qual,Central Air,Electrical,Basement Full Baths,Basement Half baths,Full Baths Above Grade,Half Baths Above Grade,Bedrooms Above Grade,Kitchens Above Grade,Kitchen Qual,Total Rooms Above Grade,Functionality,Fireplaces,Fireplce Qual,Garage Type,Year Garage,Garage Finish,Garage Cars,Garage Qual,Garage Cond,Paved Drive,Pool Qual,Fence,...,Exterior Primary_Stucco,Exterior Primary_Wood Shingles,Exterior Primary_Wood Siding,Foundation_Brick & Tile,Foundation_Cinder Block,Foundation_Slab,Foundation_Stone,Foundation_Wood,Basement Exposure_Avg Exposure,Basement Exposure_Good Exposure,Basement Exposure_Min Exposure,Basement Exposure_No Basement,Basement Finish_Avg Living Quarters,Basement Finish_Avg Rec Room,Basement Finish_Below Avg Living Quarters,Basement Finish_Good Living Quarters,Basement Finish_Low Quality,Basement Finish_No Basement,Electrical_60 AMP Fuse Box and mostly Romex wiring (Fair),Electrical_Fuse Box over 60 AMP and all Romex wiring (Average),Functionality_Major Deductions 1,Functionality_Minor Deductions 1,Functionality_Minor Deductions 2,Functionality_Moderate Deductions,Garage Type_Basement Garage,Garage Type_Built-In (Garage with Room),Garage Type_Car Port,Garage Type_Detached from home,Garage Type_More than one type of garage,Garage Type_No Garage,Garage Finish_Finished,Garage Finish_No Garage,Garage Finish_Rough Finished,Paved Drive_Dirt/Gravel,Paved Drive_Partial Pavement,Fence_Good Privacy,Fence_Good Wood,Fence_Minimum Privacy,Fence_Minimum Wood/Wire,Sale Type_Contract 15% Down payment regular terms,Sale Type_Contract Low Down,Sale Type_Contract Low Down payment and low interest,Sale Type_Contract Low Interest,Sale Type_Court Officer Deed/Estate,Sale Type_Home just constructed and sold,Sale Type_Warranty Deed - Cash,"Sale Condition_Abnormal Sale - trade, foreclosure, short sale","Sale Condition_Allocation - two linked properties with separate deeds, typically condo with a garage unit",Sale Condition_Home was not completed when last assessed (associated with New Homes),Sale Condition_Sale between family members
520,1-STORY 1946 & NEWER,Resid Low Density,61.0,7943,Regular,Level,Inside lot,Gentle,Sawyer,Adjacent Feeder St,1-family Detached,1 story,4,5,1961,1961,Gable,Composite Shingle,Vinyl Siding,192.0,2,1,Cinder Block,2,2,Min Exposure,Avg Rec Room,3,1,Standard Circuit Breakers & Romex,1,0,1,0,3,1,2,5,Typical Functionality,0,0,Attached to home,1961.0,Unfinished,1,2,2,Paved,0,No Fence,...,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
777,1-STORY 1946 & NEWER,Resid Low Density,116.0,13501,Slightly irregular,Level,Corner lot,Gentle,Somerset,Normal,1-family Detached,1 story,8,5,2006,2006,Gable,Composite Shingle,Vinyl Siding,208.0,3,2,Poured Contrete,3,2,No Exposure,Good Living Quarters,4,1,Standard Circuit Breakers & Romex,1,0,2,0,3,1,3,8,Typical Functionality,1,3,Attached to home,2006.0,Rough Finished,3,2,2,Paved,0,No Fence,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14,1-STORY 1946 & NEWER,Resid Low Density,73.0,11241,Slightly irregular,Level,Cul-de-sac,Gentle,North Ames,Normal,1-family Detached,1 story,6,7,1970,1970,Gable,Composite Shingle,Wood Siding,180.0,2,2,Cinder Block,2,2,No Exposure,Avg Living Quarters,4,1,Standard Circuit Breakers & Romex,1,0,1,0,2,1,2,5,Typical Functionality,1,2,Attached to home,1970.0,Finished,2,2,2,Paved,0,No Fence,...,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
912,2 FAMILY CONVERSION,Resid Med Density,85.0,13600,Regular,Level,Inside lot,Gentle,Old Town,Normal,2-family Conver

In [4]:
label = 'Sale Price'

# Regression target: the continuous sale price. The preprocessed file now also keeps the original
# string columns, so drop them - X_raw is the numeric feature matrix (dummies + numeric features).
X_raw = df.drop(columns=[label]).select_dtypes(exclude='object')
y = df[label].copy()

y.describe()

count      1256.0000
mean     171973.6115
std       57461.1849
min       34900.0000
25%      130500.0000
50%      161250.0000
75%      205000.0000
max      337500.0000
Name: Sale Price, dtype: float64

#### 2.1.1 Scale features

In [5]:
from sklearn.preprocessing import StandardScaler

# The data loaded from notebook 2 is already dummy-encoded, so every feature
# column is numeric. We scale the entire feature matrix to zero mean, unit variance.
std_scaler = StandardScaler()

X = pd.DataFrame(
    std_scaler.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index
)

X.head()

,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year,Year Remod Add,Masonry/Veneer Area,Exterior Qual,Exterior Cond,Basement Height,Basement Cond,Heating Qual,Central Air,Basement Full Baths,Basement Half baths,Full Baths Above Grade,Half Baths Above Grade,Bedrooms Above Grade,Kitchens Above Grade,Kitchen Qual,Total Rooms Above Grade,Fireplaces,Fireplce Qual,Year Garage,Garage Cars,Garage Qual,Garage Cond,Pool Qual,Floors,Indoor Area,Outdoor Area,Basement Finished Area Fraction,Type_1 STORY PUD,Type_1-1/2 STORY ALL AGES,Type_1-STORY 1945 & OLDER,Type_1-STORY PUD,Type_2 FAMILY CONVERSION,Type_2-1/2 STORY ALL AGES,Type_2-STORY 1945 & OLDER,Type_2-STORY 1946 & NEWER,Type_2-STORY PUD,Type_DUPLEX,Type_SPLIT FOYER,Type_SPLIT OR MULTI-LEVEL,Zoning Class_Commercial,Zoning Class_Floating Village Resid,Zoning Class_Resid High Density,Zoning Class_Resid Med Density,Lot Shape_Irregular,Lot Shape_Slightly irregular,...,Exterior Primary_Stucco,Exterior Primary_Wood Shingles,Exterior Primary_Wood Siding,Foundation_Brick & Tile,Foundation_Cinder Block,Foundation_Slab,Foundation_Stone,Foundation_Wood,Basement Exposure_Avg Exposure,Basement Exposure_Good Exposure,Basement Exposure_Min Exposure,Basement Exposure_No Basement,Basement Finish_Avg Living Quarters,Basement Finish_Avg Rec Room,Basement Finish_Below Avg Living Quarters,Basement Finish_Good Living Quarters,Basement Finish_Low Quality,Basement Finish_No Basement,Electrical_60 AMP Fuse Box and mostly Romex wiring (Fair),Electrical_Fuse Box over 60 AMP and all Romex wiring (Average),Functionality_Major Deductions 1,Functionality_Minor Deductions 1,Functionality_Minor Deductions 2,Functionality_Moderate Deductions,Garage Type_Basement Garage,Garage Type_Built-In (Garage with Room),Garage Type_Car Port,Garage Type_Detached from home,Garage Type_More than one type of garage,Garage Type_No Garage,Garage Finish_Finished,Garage Finish_No Garage,Garage Finish_Rough Finished,Paved Drive_Dirt/Gravel,Paved Drive_Partial Pavement,Fence_Good Privacy,Fence_Good Wood,Fence_Minimum Privacy,Fence_Minimum Wood/Wire,Sale Type_Contract 15% Down payment regular terms,Sale Type_Contract Low Down,Sale Type_Contract Low Down payment and low interest,Sale Type_Contract Low Interest,Sale Type_Court Officer Deed/Estate,Sale Type_Home just constructed and sold,Sale Type_Warranty Deed - Cash,"Sale Condition_Abnormal Sale - trade, foreclosure, short sale","Sale Condition_Allocation - two linked properties with separate deeds, typically condo with a garage unit",Sale Condition_Home was not completed when last assessed (associated with New Homes),Sale Condition_Sale between family members
0,-0.2235,-0.2585,0.7613,-0.5529,1.0613,0.8877,0.4679,1.1692,-0.2401,0.6908,0.0817,0.8865,0.2429,1.1657,-0.2412,0.8515,1.2412,0.1569,-0.2165,0.8301,1.0488,-0.9300,-0.9302,1.0423,0.3674,0.2403,0.2394,-0.0426,1.1590,0.1082,-0.1186,1.1368,-0.0801,-0.3369,-0.224,-0.2556,-0.1425,-0.0982,-0.2057,1.9813,-0.2036,-0.1905,-0.1272,-0.2078,-0.085,-0.2279,-0.1099,-0.3956,-0.1564,-0.7088,...,-0.1239,-0.1272,-0.3970,-0.3205,-0.8869,-0.1239,-0.0565,-0.0489,-0.4196,-0.2591,-0.2893,-0.1564,-0.4339,-0.3265,-0.3427,1.6381,-0.2336,-0.1537,-0.1272,-0.2556,-0.094,-0.1482,-0.1617,-0.0982,-0.1023,-0.2336,-0.0801,-0.6117,-0.0565,-0.22,-0.5356,-0.22,1.4957,-0.2317,-0.1304,-0.2078,-0.2036,-0.3613,-0.094,-0.0399,-0.0801,-0.0565,-0.0565,-0.1766,-0.2677,-0.0565,-0.2643,-0.085,-0.2711,-0.1239
1,0.6115,0.1289,-0.0263,2.1990,0.1464,-0.4364,-0.4856,-0.6944,-0.2401,0.6908,0.0817,0.8865,0.2429,-0.8048,4.0257,0.8515,-0.7498,0.1569,-0.2165,-0.7813,-0.2879,0.6923,0.5960,-0.0633,0.3674,0.2403,0.2394,-0.0426,-0.8628,0.0440,0.4415,0.9969,-0.0801,-0.3369,-0.224,-0.2556,-0.1425,-0.0982,-0.2057,-0.5047,-0.2036,-0.1905,-0.1272,-0.2078,-0.085,-0.2279,-0.1099,-0.3956,-0.1564,-0.7088,...,-0.1239,-0.1272,-0.3970,-0.3205,1.1275,-0.1239,-0.0565,-0.0489,-0.4196,3.8599,-0.2893,-0.1564,2.3047,-0.3265,-0.3427,-0.6105,-0.2336,-0.1537,-0.1272,-0.2556,-0.094,-0.1482,-0.1617,-0.0982,-0.1023,-0.

## 3. Modeling

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### 3.1 Parsimony (remove excess features)

Tree importance first, then iterative VIF pruning — the same recipe as the concept notebooks (`U1-4_FeatSelect-1_VIF`, `U1-4_FeatSelect-2_FeatureImportance`). For the embedded (L1) alternative see `U1-4_FeatSelect-5_Lasso`, and for projection-based parsimony see `U1-4_FeatSelect-3_PCA`.

In [7]:
from sklearn.ensemble import GradientBoostingRegressor

# Fit a Gradient Boosting model on the TRAINING data only
gb = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_leaf_nodes=10
)
gb.fit(X_train, y_train)

# Get feature importances
importances = pd.Series(
    gb.feature_importances_, index=X.columns
).sort_values(ascending=False)

importances = importances[:100]

features_gb = importances.index

X_pars = X[ features_gb ]
importances

Indoor Area                                                       3.8794e-01
Overall Qual                                                      3.6523e-01
Kitchen Qual                                                      3.1074e-02
Year                                                              2.8431e-02
Outdoor Area                                                      2.5365e-02
Year Remod Add                                                    2.0245e-02
Lot Area                                                          1.6692e-02
Type_2-STORY 1946 & NEWER                                         1.6616e-02
Basement Finished Area Fraction                                   1.4543e-02
Exterior Qual                                                     1.3595e-02
Overall Cond                                                      1.2269e-02
Garage Cars                                                       9.9069e-03
Fireplce Qual                                                     8.7152e-03

#### 3.1.1 Variance Inflation Factor

In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X_pars_with_const = sm.add_constant(X_pars)

vif = pd.Series(
    [variance_inflation_factor(X_pars_with_const.values, i) for i in range(X_pars_with_const.shape[1])],
    index=X_pars_with_const.columns
).sort_values(ascending=False)

vif[:10]

Type_1-1/2 STORY ALL AGES               14.0136
House Style_1.5 story: 2nd level fin    13.1892
Year                                    12.1482
House Style_2 story                      8.8527
Garage Cond                              8.3777
Garage Type_No Garage                    8.2237
Garage Qual                              7.9481
Type_2-STORY 1946 & NEWER                6.9410
Indoor Area                              5.2662
Year Garage                              5.2533
dtype: float64

In [9]:
# Copy dataset to avoid modifying the original
X_vif = X_pars.copy()

# Iteratively remove features with high VIF
while True:
    # Add constant for intercept
    X_vif_with_const = sm.add_constant(X_vif)
    
    # Compute VIF for all features
    vif_series = pd.Series(
        [variance_inflation_factor(X_vif_with_const.values, i) for i in range(X_vif_with_const.shape[1])],
        index=X_vif_with_const.columns
    )
    
    # Exclude constant term and get the feature with the highest VIF
    vif_series = vif_series.drop('const', errors='ignore')
    highest_vif_feature = vif_series.idxmax()
    #display(vif_series)
    
    # Break the loop if all features have VIF ≤ 10
    if vif_series.loc[highest_vif_feature] <= 10:  # Checking the first feature after 'const'
        break

    # Drop the feature with the highest VIF
    X_vif = X_vif.drop(columns=[highest_vif_feature])

    print(f"Dropped: {highest_vif_feature} (VIF={vif_series.loc[highest_vif_feature]:.2f})")
# end

vif_series.sort_values(ascending=False)[:10]

Dropped: Type_1-1/2 STORY ALL AGES (VIF=14.01)


Dropped: Year (VIF=11.96)


Garage Cond                            8.2921
House Style_2 story                    8.2662
Garage Type_No Garage                  8.1924
Garage Qual                            7.8590
Type_2-STORY 1946 & NEWER              6.5294
Indoor Area                            5.2620
Nbhd_Somerset                          5.1229
Year Garage                            4.8199
Fireplce Qual                          4.5797
Zoning Class_Floating Village Resid    4.5795
dtype: float64

### 3.2 Grid search on the selected features

Parsimony has trimmed the feature set down to the columns in `X_vif`. We now model those features
with the **same grid search used in the regression notebook** (`U1_RealEstate-3_Regression`): each
model family gets its own hyperparameter grid, `GridSearchCV` tunes it with 5-fold cross-validation
on the training set, and the best-tuned version of every family is compared on the held-out test
set. The only change from that notebook is the input matrix — here we fit on the parsimony-selected
features (`X_vif.columns`) rather than all of them.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# (name, estimator, hyperparameter grid) for each family.
# Linear regression has almost nothing to tune, so its "grid" is a single toggle.
model_families = [
    ("Linear Regression", LinearRegression(),
    {
        "fit_intercept": [True, False],
    }),
    ("KNN", KNeighborsRegressor(),
    {
        "n_neighbors": [5, 10, 20, 50],
    }),
    ("Decision Tree", DecisionTreeRegressor(random_state=42),
    {
        "min_samples_leaf": [3, 5, 10, 20],
    }),
    ("Random Forest", RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100, 200, 300],
        "min_samples_leaf": [3, 5, 10, 20],
    }),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
    }),
]

In [11]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Restrict the train/test matrices to the features that survived parsimony.
X_train_sel = X_train[X_vif.columns]
X_test_sel  = X_test[X_vif.columns]

# Plain KFold: no classes to stratify in regression.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []          # one row of metrics per model
best_models = {}      # name -> fitted best estimator for each family

for name, estimator, param_grid in tqdm(model_families, ncols=70):
    # Search this family's grid, selecting on cross-validated R^2.
    grid_search = GridSearchCV(estimator, param_grid, cv=kf,
                               scoring='r2', n_jobs=-1)
    grid_search.fit(X_train_sel, y_train)

    best = grid_search.best_estimator_
    best_models[name] = best

    # Metrics for the tuned winner, on both splits.
    y_train_pred = best.predict(X_train_sel)
    y_test_pred = best.predict(X_test_sel)
    results.append({
        "Model": name,
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
        "Test MAPE": mean_absolute_percentage_error(y_test, y_test_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Best params": grid_search.best_params_,
    })

# All results from all models, collected in one DataFrame (best test R^2 first).
results_df = pd.DataFrame(results).set_index("Model").sort_values("Test R2", ascending=False)

  0%|                                           | 0/5 [00:00<?, ?it/s]

 20%|███████                            | 1/5 [00:02<00:11,  2.92s/it]

 40%|██████████████                     | 2/5 [00:08<00:13,  4.43s/it]

 80%|████████████████████████████       | 4/5 [00:14<00:03,  3.64s/it]

100%|███████████████████████████████████| 5/5 [00:16<00:00,  2.99s/it]

100%|███████████████████████████████████| 5/5 [00:16<00:00,  3.26s/it]

In [12]:
results_df.round(2).sort_values("Test MAE")

,Test MAE,Test MAPE,Test R2,Best params
Model,,,,
Gradient Boosting,12662.85,0.08,0.91,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti..."
Random Forest,13078.62,0.09,0.90,"{'min_samples_leaf': 3, 'n_estimators': 300}"
Linear Regression,13222.95,0.09,0.91,{'fit_intercept': True}
Decision Tree,19197.27,0.12,0.79,{'min_samples_leaf': 5}
KNN,19569.64,0.13,0.79,{'n_neighbors': 5}


## Review

- **Two prunings, two different questions.** Importance asks whether a feature is *useful*; VIF asks
  whether it is *redundant given the others*. A feature can be both.
- **VIF is recomputed after every removal**, because it is a property of the whole set — removing
  one member of a collinear group often brings the rest back under threshold on its own.
- **Importance is measured on the training split only**, so the pruning decision never sees the test
  set.
- **Judge the result in dollars.** As argued in `U1_RealEstate-3_Regression`, MAE is the metric that
  answers the question a buyer asks; the target here is *no meaningful increase* in MAE on a much
  smaller feature set.
- **Parsimony buys robustness and explainability, not accuracy.** A model with a fraction of the
  columns is cheaper to fit, easier to describe to a client, and less fragile when a field goes
  missing.